# Responsible AI: Age Classification System

#### Training Data: UTKFace | Fairness Testing: FairFace
#### Goal: Build a race-agnostic age classifier for retail analytics



In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
BASE = Path.home() / "Downloads" / "responsible ai"

# UTKFace — for training
UTK_DIR = BASE / "archive" / "UTKFace"

# FairFace — for fairness testing only
FAIRFACE_TRAIN = BASE / "FairFace Race" / "train"
FAIRFACE_VAL   = BASE / "FairFace Race" / "val"

print("UTKFace exists  :", UTK_DIR.exists())
print("FairFace exists :", FAIRFACE_TRAIN.exists())

UTKFace exists  : False
FairFace exists : False



##  Data Loading


###  Load UTKFace

In [ ]:
RACE_MAP   = {0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Other"}
GENDER_MAP = {0: "Male",  1: "Female"}

rows    = []
skipped = 0

for img in UTK_DIR.iterdir():
    if not img.name.endswith(".jpg"):
        skipped += 1
        continue
    parts = img.stem.split("_")
    if len(parts) < 3:
        skipped += 1
        continue
    try:
        rows.append({
            "filepath" : str(img),
            "age"      : int(parts[0]),
            "gender"   : GENDER_MAP.get(int(parts[1]), "Unknown"),
            "race"     : RACE_MAP.get(int(parts[2]),   "Unknown"),
        })
    except ValueError:
        skipped += 1

df_utk = pd.DataFrame(rows)

print("UTKFace loaded :", len(df_utk), "images")
print("Skipped        :", skipped)
df_utk.head(3)

### Load FairFace

In [ ]:
rows = []

for race_folder in FAIRFACE_TRAIN.iterdir():
    for img in race_folder.glob("*.jpg"):
        rows.append({"filepath": str(img), "race": race_folder.name, "split": "train"})

for race_folder in FAIRFACE_VAL.iterdir():
    for img in race_folder.glob("*.jpg"):
        rows.append({"filepath": str(img), "race": race_folder.name, "split": "val"})

df_fairface = pd.DataFrame(rows)

print("FairFace loaded :", len(df_fairface), "images")
print("Races           :", sorted(df_fairface["race"].unique()))
df_fairface.head(3)

## Data Cleaning & Privacy



### Clean UTKFace

In [ ]:
df_utk_clean = df_utk.copy()

# Step 1: Remove extreme ages
df_utk_clean = df_utk_clean[df_utk_clean["age"] >= 1]
df_utk_clean = df_utk_clean[df_utk_clean["age"] <= 100]

# Step 2: Remove unknown gender
df_utk_clean = df_utk_clean[df_utk_clean["gender"] != "Unknown"]

# Step 3: Remove nulls
df_utk_clean = df_utk_clean.dropna(subset=["age", "gender", "race"])

# Step 4: Verify files exist on disk
df_utk_clean["exists"] = df_utk_clean["filepath"].apply(os.path.exists)
df_utk_clean = df_utk_clean[df_utk_clean["exists"]].drop(columns=["exists"])

# Step 5: Flag minors
df_utk_clean["is_minor"] = df_utk_clean["age"] < 18

print("UTKFace after cleaning :", len(df_utk_clean))
print("Minors (age < 18)      :", df_utk_clean["is_minor"].sum())
print("Adults                 :", (~df_utk_clean["is_minor"]).sum())
df_utk_clean.head(3)

### Clean FairFace

In [ ]:
df_fairface_clean = df_fairface.copy()

# Step 1: Verify files exist on disk
df_fairface_clean["exists"] = df_fairface_clean["filepath"].apply(os.path.exists)
df_fairface_clean = df_fairface_clean[df_fairface_clean["exists"]].drop(columns=["exists"])

# Step 2: Remove nulls
df_fairface_clean = df_fairface_clean.dropna(subset=["race"])

# Step 3: Keep only needed columns
df_fairface_clean = df_fairface_clean[["filepath", "race", "split"]]

print("FairFace after cleaning :", len(df_fairface_clean))
print("Races :", sorted(df_fairface_clean["race"].unique()))
df_fairface_clean.head(3)

### Add Age Groups to UTKFace

In [ ]:
def age_to_group(age):
    if   age <= 2:  return "0-2"
    elif age <= 9:  return "3-9"
    elif age <= 19: return "10-19"
    elif age <= 29: return "20-29"
    elif age <= 39: return "30-39"
    elif age <= 49: return "40-49"
    elif age <= 59: return "50-59"
    elif age <= 69: return "60-69"
    else:           return "70+"

df_utk_clean["age_group"] = df_utk_clean["age"].apply(age_to_group)

print("Age groups created:")
print(df_utk_clean["age_group"].value_counts().sort_index())

### Cleaning Decisions & Privacy Rationale

| Step | Decision | Reason |
|---|---|---|
| 1 | Removed ages < 1 and > 100 | Likely annotation errors |
| 2 | Removed Unknown gender | Cannot use for fairness testing |
| 3 | Removed null rows | Data integrity |
| 4 | Verified files on disk | No broken records in pipeline |
| 5 | Flagged minors (age < 18) | GDPR Article 8: children need special protection |
| 6 | Kept race for bias audit only | Race is NEVER a model input |
| 7 | Dropped Adience dataset | Too small, no race labels, rough age brackets only |

**UTKFace → Model Training**
**FairFace → Fairness Testing only**

## Exploratory Data Analysis (EDA)

In [ ]:
counts = {"UTKFace (train)": len(df_utk_clean),
          "FairFace (bias test)": len(df_fairface_clean)}

plt.figure(figsize=(6, 4))
plt.bar(counts.keys(), counts.values(), color=["coral", "steelblue"], edgecolor="black")
plt.title("Dataset Sizes")
plt.ylabel("Number of Images")
for i, v in enumerate(counts.values()):
    plt.text(i, v + 100, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

###  UTKFace Age Distribution

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df_utk_clean["age"], bins=40, color="coral", edgecolor="black")
plt.title("UTKFace — Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print("Min age :", df_utk_clean["age"].min())
print("Max age :", df_utk_clean["age"].max())
print("Mean age:", round(df_utk_clean["age"].mean(), 1))

### UTKFace Age Group Distribution

In [ ]:
plt.figure(figsize=(9, 4))
df_utk_clean["age_group"].value_counts().sort_index().plot(
    kind="bar", color="coral", edgecolor="black"
)
plt.title("UTKFace — Age Group Distribution")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### UTKFace Race Distribution

In [ ]:
plt.figure(figsize=(7, 4))
df_utk_clean["race"].value_counts().plot(
    kind="bar", color="steelblue", edgecolor="black"
)
plt.title("UTKFace — Race Distribution")
plt.xlabel("Race")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### UTKFace Gender Distribution

In [ ]:
plt.figure(figsize=(5, 4))
df_utk_clean["gender"].value_counts().plot(
    kind="bar", color=["steelblue", "coral"], edgecolor="black"
)
plt.title("UTKFace — Gender Distribution")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### FairFace Race Distribution

In [ ]:
plt.figure(figsize=(9, 4))
df_fairface_clean["race"].value_counts().plot(
    kind="bar", color="steelblue", edgecolor="black"
)
plt.title("FairFace — Race Distribution (Fairness Test Set)")
plt.xlabel("Race")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Bias Check: Age by Race

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_utk_clean, x="race", y="age", palette="Set2")
plt.title("UTKFace — Age Distribution per Race\n(Key Bias Check)")
plt.xlabel("Race")
plt.ylabel("Age")
plt.tight_layout()
plt.show()

## Sample Images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

img1 = mpimg.imread(df_utk_clean["filepath"].iloc[0])
axes[0].imshow(img1)
axes[0].set_title(f"UTKFace Sample\nAge: {df_utk_clean['age'].iloc[0]}, "
                  f"Gender: {df_utk_clean['gender'].iloc[0]}, "
                  f"Race: {df_utk_clean['race'].iloc[0]}")
axes[0].axis("off")

img2 = mpimg.imread(df_fairface_clean["filepath"].iloc[0])
axes[1].imshow(img2)
axes[1].set_title(f"FairFace Sample\nRace: {df_fairface_clean['race'].iloc[0]}")
axes[1].axis("off")

plt.suptitle("Sample Images", fontsize=13)
plt.tight_layout()
plt.show()

## EDA Summary (Markdown)

### EDA Key Findings

| Finding | Impact on Model |
|---|---|
| Young faces (20-30) dominate UTKFace | Model may underperform on elderly |
| White faces outnumber others in UTKFace | Potential accuracy gap for other races |
| FairFace balanced across 7 races | Good fairness test set |
| Minors present in UTKFace | Documented — special care in deployment |
| Age 70+ very underrepresented | Use class weights during training |

**Next Step → Model Building on UTKFace**

# Week 2 Goals Mapped Out

### Step 1 — Baseline Model


### Framework: Keras (TensorFlow 2.21)
### Model: MobileNetV2 (pretrained, fine-tuned for age classification)
### Input: Face image → Output: Age group



### Step 2 — Risk Analysis

Goes in a Markdown + code section
Cover 5 risks minimum:

Demographic bias
Misclassification of minors
Adversarial attacks
Data privacy
Model failure in bad lighting




### Step 3 — Fairness Analysis

Run trained model on FairFace
Measure accuracy per race
Plot results
Conclude: is the model fair or not?

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

### Age Groups (9 classes)
| Class | Age Range |
|---|---|
| 0 | 0-2 |
| 1 | 3-9 |
| 2 | 10-19 |
| 3 | 20-29 |
| 4 | 30-39 |
| 5 | 40-49 |
| 6 | 50-59 |
| 7 | 60-69 |
| 8 | 70+ |

### Map Age to Class Number

In [ ]:
# Map age group string to number
AGE_GROUP_MAP = {
    "0-2"   : 0,
    "3-9"   : 1,
    "10-19" : 2,
    "20-29" : 3,
    "30-39" : 4,
    "40-49" : 5,
    "50-59" : 6,
    "60-69" : 7,
    "70+"   : 8
}

NUM_CLASSES = len(AGE_GROUP_MAP)

# Add numeric label to UTKFace dataframe
df_utk_clean["label"] = df_utk_clean["age_group"].map(AGE_GROUP_MAP)

print("Label mapping:")
for k, v in AGE_GROUP_MAP.items():
    print(f"  {v} → {k}")

print("\nLabel counts:")
print(df_utk_clean["label"].value_counts().sort_index())

### Train / Validation / Test Split

In [ ]:
# Split: 70% train, 15% val, 15% test
# Stratify on label so each split has all age groups

train_df, temp_df = train_test_split(
    df_utk_clean,
    test_size=0.30,
    stratify=df_utk_clean["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train size      :", len(train_df))
print("Validation size :", len(val_df))
print("Test size       :", len(test_df))

### Image Settings

In [ ]:
IMG_SIZE   = 96       # resize all images to 96x96
BATCH_SIZE = 32       # 32 images per batch
EPOCHS     = 10       # train for 10 rounds

print("Image size :", IMG_SIZE)
print("Batch size :", BATCH_SIZE)
print("Epochs     :", EPOCHS)

### Image Data Generators

In [ ]:
# Train generator — with augmentation to improve generalization
train_gen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1
)

# Val and test generators — no augmentation, only rescale
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

print("Image generators ready")

### Load Images Function

In [ ]:
import numpy as np
from PIL import Image

def load_images(dataframe, img_size):
    images = []
    labels = []

    for _, row in dataframe.iterrows():
        try:
            img = Image.open(row["filepath"])
            img = img.convert("RGB")
            img = img.resize((img_size, img_size))
            img = np.array(img) / 255.0
            images.append(img)
            labels.append(row["label"])
        except Exception:
            continue

    images = np.array(images)
    labels = to_categorical(labels, num_classes=NUM_CLASSES)
    return images, labels

print("Loading train images...")
X_train, y_train = load_images(train_df, IMG_SIZE)

print("Loading val images...")
X_val, y_val = load_images(val_df, IMG_SIZE)

print("Loading test images...")
X_test, y_test = load_images(test_df, IMG_SIZE)

print("\nTrain shape :", X_train.shape)
print("Val shape   :", X_val.shape)
print("Test shape  :", X_test.shape)

## Build the Model

In [ ]:
# Use MobileNetV2 pretrained on ImageNet
# We freeze the base and only train the top layers

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze base model weights
base_model.trainable = False

# Add our own classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

print("Model built successfully")
print("Total layers    :", len(model.layers))
print("Output classes  :", NUM_CLASSES)

### Compile the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled")
model.summary()

### Train the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("Training complete")

### Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history["accuracy"],     label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Loss
axes[1].plot(history.history["loss"],     label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.suptitle("Training History", fontsize=13)
plt.tight_layout()
plt.show()

### Evaluate on Test Set

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("=== TEST SET RESULTS ===")
print(f"Test Accuracy : {test_accuracy:.2%}")
print(f"Test Loss     : {test_loss:.4f}")

### Save the Model

In [ ]:
# model.save("age_classifier_baseline.keras")
# print("Model saved as age_classifier_baseline.keras")

### Baseline Model Summary
- Architecture : MobileNetV2 pretrained on ImageNet
- Input        : 96x96 RGB face image
- Output       : 9 age group classes
- Optimizer    : Adam
- Loss         : Categorical Crossentropy
- Epochs       : 10
- Race/Gender  : NOT used as input features (race-agnostic by design)

**Next → Section 5: Fairness Analysis on FairFace**